# Indian Insurance Policy Multimodal RAG with Gemini 2.5 Vision

This showcase demonstrates an enterprise-grade **Structure-Aware Multimodal RAG System** designed for complex Indian Health Insurance policies governed by the **Insurance Regulatory and Development Authority of India (IRDAI)**.

### Key Architecture Innovations:
1. **Structure-Aware Pre-Chunking Layout Analysis**: Partitions contracts across 9 canonical IRDAI sections.
2. **Parent-Child Hierarchical Chunking + Cross-Reference Graph**: Resolves non-linear clause linkages (`Excl01` to `Excl18`) and enforces legal precedence (`Policy Wordings & Schedule > CIS Summary`).
3. **Gemini 2.5 Vision Table Extraction**: 200-DPI visual crops of complex benefit grids paired with synthetic question-answer pairs.
4. **Hybrid Retrieval**: Local Qdrant Dense Vectors + BM25Okapi Sparse fusion via Reciprocal Rank Fusion (RRF).
5. **Pre-Retrieval Intent Routing + Bounded CRAG**: Decomposes multi-clause queries and escalates to Gemini 2.5 Pro when needed.
6. **Calibrated Confidence & IRDAI Abstention**: Deterministic abstention below 0.70 confidence.
7. **DPDP Act 2023 Compliance & FinOps**: Automatic Aadhaar/PAN masking and real-time USD/INR cost tracking.


In [1]:
# Step 1: Initialize Pipeline and Generate Sample IRDAI Policies
from pathlib import Path
from insurance_mmrag.pipeline import create_default_pipeline
from insurance_mmrag.ingestion.sample_policy_factory import (
    create_arogya_sanjeevani_pdf,
    create_saral_suraksha_bima_pdf,
)

data_dir = Path("data/sample_policies")
data_dir.mkdir(parents=True, exist_ok=True)
arogya_pdf = create_arogya_sanjeevani_pdf(data_dir / "arogya_sanjeevani.pdf")
saral_pdf = create_saral_suraksha_bima_pdf(data_dir / "saral_suraksha.pdf")

pipeline = create_default_pipeline(use_stub=True)
doc_arogya = pipeline.ingest_pdf(arogya_pdf, doc_id="arogya_sanjeevani")
doc_saral = pipeline.ingest_pdf(saral_pdf, doc_id="saral_suraksha")

print(f"Indexed {doc_arogya.doc_id}: {len(doc_arogya.parents)} parents, {len(doc_arogya.children)} children, {len(doc_arogya.tables)} tables")
print(f"Indexed {doc_saral.doc_id}: {len(doc_saral.parents)} parents, {len(doc_saral.children)} children")

In [2]:
# Step 2: Query 1 — Complex Waiting Period & Table Limit Query
query_1 = "What is the waiting period for cataract surgery under this policy?"
resp_1 = pipeline.query(query_1)

print("=== GROUNDED ANSWER ===")
print(resp_1.answer)
print(f"\nConfidence Score: {resp_1.confidence_score * 100:.1f}%")
print(f"Coverage Status: {resp_1.coverage_summary.coverage_status}")
print(f"Waiting Period: {resp_1.coverage_summary.waiting_period_details}")
print(f"Sub-Limits: {resp_1.coverage_summary.applicable_sub_limits}")
print(f"Legal Precedence: {resp_1.precedence_check.rule_applied}")
print(f"\nCitations ({len(resp_1.citations)}): {[c.clause_code or c.section_title for c in resp_1.citations]}")

In [3]:
# Step 3: Query 2 — Adversarial Out-of-Scope Trap (Deterministic Abstention < 0.70)
query_adv = "Does this policy cover pet veterinary dental implants in Antarctica?"
resp_adv = pipeline.query(query_adv)

print("=== ADVERSARIAL ABSTENTION TEST ===")
print(f"Abstained: {resp_adv.abstained}")
print(f"Confidence: {resp_adv.confidence_score * 100:.1f}%")
print(f"Abstention Reason: {resp_adv.abstention_reason}")
print(f"Disclaimer: {resp_adv.answer}")

In [4]:
# Step 4: Query 3 — DPDP Act 2023 PII Masking Verification
query_pii = "My Aadhaar is 1234 5678 9012 and PAN ABCDE1234F. What is the room rent limit?"
resp_pii = pipeline.query(query_pii)

print("=== DPDP ACT 2023 PII REDACTION ===")
print(f"Original: {resp_pii.intent.original_query}")
print(f"Redacted: {resp_pii.intent.redacted_query}")
print(f"Masked Categories: {resp_pii.pii_redactions_applied}")

In [5]:
# Step 5: Quantitative RAG Triad Scorecard Benchmark
from insurance_mmrag.evals.benchmark_runner import BenchmarkRunner

runner = BenchmarkRunner(pipeline)
card = runner.run_benchmark()

print("=== QUANTITATIVE RAG TRIAD SCORECARD ===")
print(f"Context Recall @ k: {card.context_recall_at_k * 100:.1f}%")
print(f"Answer Faithfulness: {card.answer_faithfulness * 100:.1f}%")
print(f"Abstention F1: {card.abstention_f1 * 100:.1f}%")
print(f"Table Numerical Exactness: {card.table_numerical_exactness * 100:.1f}%")
print(f"Total Test Cases: {card.total_cases}")

In [6]:
# Step 6: FinOps Telemetry & OpenTelemetry Execution Waterfall
finops = pipeline.get_finops_summary()
print("=== FINOPS TELEMETRY ===")
print(f"Total Spend (USD): ${finops.total_cost_usd:.4f}")
print(f"Total Spend (INR): ₹{finops.total_cost_inr:.2f}")
print(f"Cache Hit Rate: {finops.cache_hit_rate_pct:.1f}%")

print("\n=== OPENTELEMETRY EXECUTION SPANS (Query 1) ===")
for span in resp_1.trace_spans:
    print(f"[{span.status}] {span.name:<32} Offset: {span.start_offset_ms:>6.2f}ms | Duration: {span.duration_ms:>6.2f}ms")